# Colab 11c — Natural-only training, V1 only

## Hypothesis under test

**Iter 1 (colab11) and iter 2 (colab11b) both failed natural-low V1.** Predictions clustered at the natural-pool mean instead of refining within the [0.01, 0.33] band. We attributed this to a **synthetic-shortcut**: synthetic perturbation pairs share contiguous identical substrings, natural pairs don't, the network learned to detect that procedural distinction instead of approximating Levenshtein.

**This iter (11c) tests the simplest possible counter-hypothesis:** *if we remove synthetic data entirely and train on natural pairs only, can the network approximate `normLev` over the natural label range [0.01, 0.87]?*

Two outcomes:
- **V1 r > 0 across natural held-out** → synthetic was indeed the cause. The shortcut disappears when the input distinction (synthetic vs natural procedure) disappears. Path forward: stay with natural-only, scale up data if helpful, then cross-rep test.
- **V1 r still flat on natural held-out** → information-theoretic limit. The Levenshtein value within [0.01, 0.33] band on real S20 proteins isn't recoverable from AA inputs alone (sequences are too uniform-random in that band to discriminate). Path forward: accept the limit, focus on the [0.30, 0.87] band where Lev is biologically meaningful.

## Setup

- **Training:** 1,507 natural-high pairs (all available with both ends in train70) + 1,507 random natural-low pairs (sampled from 12,177 available). Total 3,014 training pairs. **No synthetic.**
- **Architecture:** unchanged from colab11b (embed_dim=32, hidden2=64, out_dim=128, ~1.6M params). Bigger-encoder test deferred to a later iter if this one is inconclusive.
- **Held-out V1:** natural-low (2,313 pairs from test30) + natural-high (294 pairs from test30) + a combined panel.
- **V2 dropped from this iter.** Function approximation is the focus; biological-relevance check comes after V1 is satisfied.
- **Coverage caveat:** training labels span [0.01, 0.87]. No data above 0.87 (~near-identical pairs). V1 metrics are valid over [0.01, 0.87] only.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN_KEEP = 50
MAX_LEN_KEEP = 200
MAX_LEN = 200

# Smaller dataset → can afford more epochs; we'll also watch for overfitting
NUM_EPOCHS = 40
BATCH_SIZE = 128

## 3. Helpers

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def is_standard(seq):
    return all(c in AA_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

## 4. Load proteins

In [ ]:
def load_protein_csv(path):
    df = pd.read_csv(path, compression='gzip').dropna(subset=['aa_seq', 'SuperFamily'])
    df['aa_seq'] = df['aa_seq'].astype(str)
    df = df[df['aa_seq'].str.len().between(MIN_LEN_KEEP, MAX_LEN_KEEP)]
    df = df[df['aa_seq'].apply(is_standard)]
    return df

train_df = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))

train_seqs = {r['domain_id']: r['aa_seq'] for _, r in train_df.iterrows()}
test_seqs  = {r['domain_id']: r['aa_seq'] for _, r in test_df.iterrows()}

print(f'train: {len(train_seqs)} proteins')
print(f'test:  {len(test_seqs)} proteins')

## 5. Load pair files

In [ ]:
PAIR_COLS = ['domain_p2', 'domain_p1', 'ss_score', 'aa_score',
             'TM_min', 'TM_max', 'cath_superFamily']

def load_pairs(path):
    df = pd.read_csv(path, compression='gzip', header=None, names=PAIR_COLS)
    df['aa_score'] = pd.to_numeric(df['aa_score'], errors='coerce')
    return df.dropna(subset=['aa_score']).reset_index(drop=True)

pairs_sample = load_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'))
pairs_high   = load_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'))

print(f'pairs_sample: {len(pairs_sample)} rows, aa_score range [{pairs_sample.aa_score.min():.3f}, {pairs_sample.aa_score.max():.3f}]')
print(f'pairs_high:   {len(pairs_high)} rows, aa_score range [{pairs_high.aa_score.min():.3f}, {pairs_high.aa_score.max():.3f}]')

## 6. Filter pairs to train/test ID sets

In [ ]:
def filter_pairs(df, id_set):
    return df[df['domain_p1'].isin(id_set) & df['domain_p2'].isin(id_set)].reset_index(drop=True)

train_ids = set(train_seqs.keys())
test_ids  = set(test_seqs.keys())

natural_low_train  = filter_pairs(pairs_sample, train_ids)
natural_low_test   = filter_pairs(pairs_sample, test_ids)
natural_high_train = filter_pairs(pairs_high,   train_ids)
natural_high_test  = filter_pairs(pairs_high,   test_ids)

print(f'natural-low train pool:  {len(natural_low_train)}')
print(f'natural-low test pool:   {len(natural_low_test)}')
print(f'natural-high train pool: {len(natural_high_train)}')
print(f'natural-high test pool:  {len(natural_high_test)}')

## 7. Build training set — count-matched, natural only

Take all `natural_high_train` pairs. Sample the same count from `natural_low_train`. Combine, shuffle, plot the resulting label distribution.

In [ ]:
def df_to_pairs(df, seq_dict):
    pairs = []
    for _, r in df.iterrows():
        a = seq_dict.get(r['domain_p1']); b = seq_dict.get(r['domain_p2'])
        if a and b:
            pairs.append((a, b, float(r['aa_score'])))
    return pairs

n_high_train = len(natural_high_train)
low_sample = natural_low_train.sample(n=n_high_train, random_state=SEED).reset_index(drop=True)

train_high_pairs = df_to_pairs(natural_high_train, train_seqs)
train_low_pairs  = df_to_pairs(low_sample, train_seqs)
train_pairs = train_high_pairs + train_low_pairs
rng.shuffle(train_pairs)

test_low_pairs  = df_to_pairs(natural_low_test,  test_seqs)
test_high_pairs = df_to_pairs(natural_high_test, test_seqs)

print(f'train pairs: {len(train_high_pairs)} natural-high + {len(train_low_pairs)} natural-low = {len(train_pairs)} total')
print(f'held-out pairs: {len(test_low_pairs)} natural-low + {len(test_high_pairs)} natural-high')

# Label distribution plot
plt.figure(figsize=(10, 3))
plt.hist([np.array([p[2] for p in train_low_pairs]),
          np.array([p[2] for p in train_high_pairs])],
         bins=40, stacked=True,
         label=[f'natural-low ({len(train_low_pairs)})', f'natural-high ({len(train_high_pairs)})'],
         edgecolor='k', alpha=0.75)
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title('Iter 11c training label distribution — natural only, count-matched')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 8. Dataset / Architecture (unchanged from colab11b)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return encode_pad(a), encode_pad(b), torch.tensor(label, dtype=torch.float32)

train_dl = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True)

class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x * mask.unsqueeze(1)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
    def encode(self, x): return self.encoder(x)

model = SiameseNetwork().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 9. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = F.mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} — MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Iter 11c training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final MSE: {losses[-1]:.5f}')

## 10. V1 — natural held-out only

Three panels:
- **Natural-low** (true [0.01, 0.33]): the iter 1/2 failure zone.
- **Natural-high** (true [0.30, 0.87]): the structurally meaningful zone.
- **Combined** (all natural held-out): single global Pearson over the full natural label range.

In [ ]:
def predict_pairs(pairs, batch_size=512):
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

true_low,  pred_low  = predict_pairs(test_low_pairs)
true_high, pred_high = predict_pairs(test_high_pairs)
true_all   = np.concatenate([true_low, true_high])
pred_all   = np.concatenate([pred_low, pred_high])

def report(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    print(f'{name:>22}: n={len(true_v):>5}  Pearson={pr:+.3f}  Spearman={sr:+.3f}')
    return pr, sr

print('=== V1 metrics (natural held-out) ===')
report('natural-low',      true_low,  pred_low)
report('natural-high',     true_high, pred_high)
report('combined natural', true_all,  pred_all)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_low,  pred_low,  'Natural-low held-out'),
    (true_high, pred_high, 'Natural-high held-out'),
    (true_all,  pred_all,  'Combined natural held-out'),
]):
    if len(true_v) < 10:
        ax.set_title(f'{title}\n(too few pairs)'); continue
    ax.scatter(true_v, pred_v, alpha=0.15, s=6)
    ax.plot([0, 1], [0, 1], 'r-', linewidth=2, label='y = x')
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    ax.set_title(f'{title}\nn={len(true_v)}  r={pr:+.3f}  ρ={sr:+.3f}')
    ax.set_xlabel('True normLev'); ax.set_ylabel('Predicted')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Reading the result

**Compare iter 11c combined V1 r against iter 11b iters' results:**

| Iter | Synthetic in training? | Natural-low r | Natural-high r |
|---|---|---|---|
| 11 (iter 1) | yes (30K) | −0.22 | 0.108 |
| 11b (iter 2) | yes (10K) | −0.19 | +0.30 |
| 11c (this iter) | **no** | ? | ? |

- **If natural-low r jumps positive** → synthetic was the cause. Path forward: scale up natural-only training (more pairs, possibly bigger encoder) and run cross-rep V1 on SS sequences.
- **If natural-low r stays flat** → information limit confirmed. The Levenshtein value within [0.01, 0.33] is essentially noise about AA composition for redundancy-reduced proteins. The thesis result is then: *the model approximates `normLev` over the structurally meaningful range [0.30, 0.87]; below that, the function is recoverable in principle but the input signal is too weak in the redundancy-reduced regime*.
- **If natural-high r is much better than iter 11b's 0.30** → the model's performance was being dragged down by trying to fit two different distributions simultaneously. Removing synthetic helps even if natural-low remains hard.